In [ ]:

import sys, os, json, time
sys.path.insert(0, os.path.abspath('..'))

import importlib
import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

import data_pipeline.config_variables as _cv
importlib.reload(_cv)
from data_pipeline.config_variables import DATA_FOLDER

import data_pipeline.config_cluster as _cc
importlib.reload(_cc)
from data_pipeline.config_cluster import (
    N_CLUSTERS_LOCAL, TEST_MODE, TEST_LA_CODES, WAVE, HIERARCHICAL_CLUSTER
)

import data_pipeline.helpers.cluster_summary as _cs
importlib.reload(_cs)

# ── Config ────────────────────────────────────────────────────────────────────
COVERAGE_TARGET       = 0.80
MAX_PIDPS_THRESHOLD   = 2000
MAX_PROFILES_PER_CALL = 100
MODEL                 = "gpt-4o"
RETRY_DELAY           = 10

hier_col = f"{WAVE}_{HIERARCHICAL_CLUSTER}" if HIERARCHICAL_CLUSTER else None

print(f"WAVE={WAVE}  HIERARCHICAL_CLUSTER={HIERARCHICAL_CLUSTER}  MODEL={MODEL}")
print(f"COVERAGE_TARGET={COVERAGE_TARGET:.0%}  MAX_PIDPS_THRESHOLD={MAX_PIDPS_THRESHOLD}  MAX_PROFILES_PER_CALL={MAX_PROFILES_PER_CALL}")

# ── Profile encoding (compact numeric codes instead of NL prose) ──────────────
# Format: age|sex|eth|edu|emp|mar|ten|hh|health  (? = unknown/missing)
_PROFILE_COLS = [
    (f"{WAVE}_doby_dv_eng", "age"),
    (f"{WAVE}_sex_dv",      "sex"),
    (f"{WAVE}_racel_dv",    "eth"),
    (f"{WAVE}_hiqual_dv",   "edu"),
    (f"{WAVE}_jbstat",      "emp"),
    (f"{WAVE}_marstat_dv",  "mar"),
    (f"{WAVE}_tenure_dv",   "ten"),
    (f"{WAVE}_hhtype_dv",   "hh"),
    (f"{WAVE}_scsf1",       "health"),
]

_CODEBOOK_STR = """\
VARIABLE CODEBOOK — each profile row: age|sex|eth|edu|emp|mar|ten|hh|health  (? = unknown)
  age:    years (numeric)
  sex:    1=Male 2=Female
  eth:    1=White British  2=White Irish  4=White Other  5=Mixed White/Black Carib  7=Mixed White/Asian  8=Mixed Other  9=Asian Indian  10=Asian Pakistani  11=Asian Bangladeshi  12=Asian Chinese  14=Black Caribbean  15=Black African  17=Arab  97=Other
  edu:    1=Degree  2=Other higher  3=A-level  4=GCSE  5=Other qual  9=No qual
  emp:    1=Self-employed  2=Employed  3=Unemployed  4=Retired  5=Maternity  6=Family care  7=Student  8=LT sick/disabled  97=Other
  mar:    1=Married  2=Cohabiting  3=Widowed  4=Divorced  5=Separated  6=Single
  ten:    1=Owned outright  2=Mortgage  3=Council rent  4=Housing assoc  5=Employer rent  6=Private unfurn  7=Private furn
  hh:     1=Solo M 65+  2=Solo F 60+  3=Solo adult  4=Single parent 1c  5=Single parent 2+c  6=Couple no child  8=Couple (pensionable age)  10=Couple+1c  11=Couple+2c  12=Couple+3+c  16=2 adults  17=2 adults (pensionable)  18=2 adults+child  19=3+ couple  22=3+ no couple
  health: 1=Excellent  2=Very good  3=Good  4=Fair  5=Poor"""

def _encode_profile(row, avail_cols: list) -> str:
    parts = []
    for col, _ in _PROFILE_COLS:
        if col not in avail_cols:
            parts.append("?")
            continue
        v = row[col]
        try:
            fv = float(v)
            parts.append("?" if (pd.isna(fv) or fv < 0) else str(int(fv)))
        except (TypeError, ValueError):
            parts.append("?")
    return "|".join(parts)

# ── Paths ─────────────────────────────────────────────────────────────────────
NL_PROFILE = Path(f"../{DATA_FOLDER}/5_add_nl_strings/{WAVE}_with_nl_profile.pkl")
LA_COUNTS  = Path(f"../{DATA_FOLDER}/10_synthetic_population/pidp_la_counts.csv")
API_OUT    = Path("../api/data/clusters/local_llm_clusters.csv")

for p in (NL_PROFILE, LA_COUNTS):
    if not p.exists():
        raise FileNotFoundError(f"{p} — check pipeline prerequisites.")

API_OUT.parent.mkdir(parents=True, exist_ok=True)

# ── OpenAI ────────────────────────────────────────────────────────────────────
load_dotenv(Path("..") / ".env")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY not set. Add to .env or export in shell.")
client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI client ready.")

# ── Load data ─────────────────────────────────────────────────────────────────
print("\nLoading NL profiles …", flush=True)
df_nl = pd.read_pickle(NL_PROFILE)
df_nl["pidp"] = pd.to_numeric(df_nl["pidp"], errors="coerce").astype("int64")
print(f"  {len(df_nl):,} respondents loaded.", flush=True)

# Determine which profile feature cols exist (doby_dv_eng may not always be present)
avail_profile_cols = [col for col, _ in _PROFILE_COLS if col in df_nl.columns]
missing_cols = [col for col, _ in _PROFILE_COLS if col not in df_nl.columns]
if missing_cols:
    print(f"  Warning: missing profile columns (will encode as ?): {missing_cols}", flush=True)
print(f"  Profile columns available: {len(avail_profile_cols)}/{len(_PROFILE_COLS)}", flush=True)

print("Loading LA counts …", flush=True)
df_la = pd.read_csv(LA_COUNTS, dtype={"pidp": "int64", "ladcd": str, "ladnm": str, "n": "int64"})
print(f"  {len(df_la):,} rows, {df_la['ladcd'].nunique():,} LAs loaded.", flush=True)

if TEST_MODE and TEST_LA_CODES:
    df_la = df_la[df_la["ladcd"].isin(TEST_LA_CODES)].copy()
    print(f"  TEST MODE — {df_la['ladcd'].nunique()} LA(s): "
          f"{sorted(df_la['ladnm'].unique().tolist())}", flush=True)

keep_cols = ["pidp"] + avail_profile_cols
if hier_col:
    if hier_col not in df_nl.columns:
        raise ValueError(f"hier_col '{hier_col}' not found in NL profile table")
    if hier_col not in keep_cols:
        keep_cols.append(hier_col)

df_merged = df_la.merge(df_nl[keep_cols], on="pidp", how="inner")
if hier_col:
    df_merged.rename(columns={hier_col: "group"}, inplace=True)
print(f"  Merged: {len(df_merged):,} rows", flush=True)

# Build compact profile string per row
print("  Building compact profile encodings …", flush=True)
df_merged["compact_profile"] = df_merged[avail_profile_cols].apply(
    lambda row: _encode_profile(row, avail_profile_cols), axis=1
)
print(f"  Unique compact profiles: {df_merged['compact_profile'].nunique():,}", flush=True)

# ── 80% coverage per LA × group ───────────────────────────────────────────────
def pidps_for_coverage(rows: pd.DataFrame, target: float) -> int:
    sorted_n = rows["n"].sort_values(ascending=False).values
    total = sorted_n.sum()
    if total == 0:
        return 0
    return int(np.searchsorted(np.cumsum(sorted_n), target * total)) + 1

group_by = ["ladcd", "ladnm", "group"] if hier_col else ["ladcd", "ladnm"]

coverage_rows = []
for keys, grp in df_merged.groupby(group_by):
    ladcd, ladnm, *rest = keys if isinstance(keys, tuple) else (keys,)
    coverage_rows.append({
        "ladcd": ladcd, "ladnm": ladnm,
        "group": rest[0] if rest else None,
        "unique_pidps": int(grp["pidp"].nunique()),
        "pidps_80pct": pidps_for_coverage(grp, COVERAGE_TARGET),
    })
df_coverage = pd.DataFrame(coverage_rows)

print(f"\n── 80% coverage table ──")
print(df_coverage[["ladnm", "group", "unique_pidps", "pidps_80pct"]].to_string(index=False))

eligible = df_coverage[df_coverage["pidps_80pct"] < MAX_PIDPS_THRESHOLD]
print(f"\nEligible LA×groups (pidps_80pct < {MAX_PIDPS_THRESHOLD}): {len(eligible)}")

if eligible.empty:
    raise RuntimeError(
        f"No LA×groups have pidps_80pct < {MAX_PIDPS_THRESHOLD}. "
        f"Raise MAX_PIDPS_THRESHOLD (current min is {int(df_coverage['pidps_80pct'].min())}) "
        f"or add more TEST_LA_CODES."
    )

# ── LLM helpers ───────────────────────────────────────────────────────────────
_SYSTEM = (
    "You are a social researcher analysing UK survey respondents who represent a "
    "local population. Each profile is encoded as numeric codes (see codebook in the user message). "
    "Each profile is followed by a weight — how many people in that local area that respondent represents. "
    "Read all profiles and group them into exactly {k} meaningful, distinct clusters, "
    "weighting by population weight. "
    "Give each cluster a vivid, specific name — 'Young Urban Renters' beats 'Group A'.\n\n"
    "IMPORTANT: Your response must be a raw JSON object only. "
    "Do NOT wrap it in code fences (no ```json or ```). "
    "Do NOT include any explanation or text outside the JSON object."
)

def _call_llm(profiles_weights: list, k: int, context: str) -> dict:
    lines = "\n".join(f"{i}. [weight:{w}]  {p}" for i, (p, w) in enumerate(profiles_weights))
    system_msg = _SYSTEM.format(k=k)
    prompt = (
        f"Context: {context}\n\n"
        f"{_CODEBOOK_STR}\n\n"
        f"Below are {len(profiles_weights)} respondent profiles "
        f"(numbered 0–{len(profiles_weights)-1}):\n\n{lines}\n\n"
        f"Return a raw JSON object with exactly these two keys — no code fences, no extra text:\n"
        f'  "clusters": array of exactly {k} cluster name strings\n'
        f'  "assignments": array of exactly {len(profiles_weights)} integers (0–{k-1}), '
        f"one per profile in the same order."
    )
    print(f"    {len(profiles_weights)} unique profiles, ~{len(prompt)//4:,} tokens", flush=True)
    print(f"\n── SYSTEM PROMPT ──\n{system_msg}", flush=True)
    print(f"\n── USER PROMPT ──\n{prompt}\n──────────────────\n", flush=True)
    for attempt in range(3):
        try:
            t0 = time.time()
            resp = client.chat.completions.create(
                model=MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user",   "content": prompt},
                ],
                temperature=0.3,
            )
            u = resp.usage
            print(f"    {time.time()-t0:.1f}s  in={u.prompt_tokens:,}  out={u.completion_tokens:,} tokens",
                  flush=True)
            parsed = json.loads(resp.choices[0].message.content)
            if len(parsed["clusters"]) != k:
                raise ValueError(f"Expected {k} clusters, got {len(parsed['clusters'])}")
            if len(parsed["assignments"]) != len(profiles_weights):
                raise ValueError(f"Expected {len(profiles_weights)} assignments, got {len(parsed['assignments'])}")
            return parsed
        except Exception as exc:
            print(f"    attempt {attempt+1} failed: {exc}", flush=True)
            if attempt < 2:
                time.sleep(RETRY_DELAY)
    raise RuntimeError("LLM call failed after 3 attempts.")

# ── Run LLM per eligible LA × group ──────────────────────────────────────────
print(f"\nStarting LLM calls for {len(eligible)} LA×group(s) …", flush=True)
pidp_level_rows = []

for _, meta in eligible.iterrows():
    ladcd_val = meta["ladcd"]
    ladnm_val = meta["ladnm"]
    group_val = meta.get("group")

    mask = df_merged["ladcd"] == ladcd_val
    if group_val is not None:
        mask &= df_merged["group"] == group_val
    slice_df = df_merged[mask].copy()

    # Aggregate to pidp level (sum weights across synthetic population rows)
    pidp_weights = (
        slice_df.groupby("pidp", sort=False)
        .agg(n=("n", "sum"), compact_profile=("compact_profile", "first"))
        .reset_index()
        .sort_values("n", ascending=False)
    )

    # Select top pidps that cover 80% of population
    total_n = pidp_weights["n"].sum()
    cutoff  = int(np.searchsorted(pidp_weights["n"].cumsum().values, 0.80 * total_n)) + 1
    top     = pidp_weights.iloc[:cutoff].copy()

    # Deduplicate by compact profile string — sum weights for identical encodings
    deduped = (
        top.groupby("compact_profile", sort=False)
        .agg(n=("n", "sum"))
        .reset_index()
        .sort_values("n", ascending=False)
        .reset_index(drop=True)
    )

    # Cap at MAX_PROFILES_PER_CALL (already sorted by weight)
    if len(deduped) > MAX_PROFILES_PER_CALL:
        print(f"  Capping from {len(deduped)} → {MAX_PROFILES_PER_CALL} unique profiles", flush=True)
        deduped = deduped.iloc[:MAX_PROFILES_PER_CALL].copy()

    k = min(N_CLUSTERS_LOCAL, len(deduped))
    context = f"{ladnm_val} — {group_val if group_val else 'all respondents'}"

    print(f"\n[{ladnm_val} / {group_val or '—'}]  "
          f"{len(top)} pidps → {len(deduped)} unique profiles → {k} clusters …",
          flush=True)

    result        = _call_llm(list(zip(deduped["compact_profile"], deduped["n"].astype(int))), k, context)
    cluster_names = result["clusters"]

    # Map cluster assignment back to each row in `top` via compact profile
    profile_to_cluster = {
        row["compact_profile"]: min(int(result["assignments"][i]), len(cluster_names) - 1)
        for i, row in deduped.iterrows()
    }

    for _, row in top.iterrows():
        ci = profile_to_cluster.get(row["compact_profile"], 0)
        pidp_level_rows.append({
            "ladcd":         ladcd_val,
            "ladnm":         ladnm_val,
            "group":         group_val,
            "pidp":          int(row["pidp"]),
            "n_sipher_rows": int(row["n"]),
            "cluster":       ci + 1,
            "tribe_label":   cluster_names[ci],
        })

df_pidp = pd.DataFrame(pidp_level_rows)

# ── Build cluster summary (matches local_values_clusters.csv format) ──────────
print("\nBuilding cluster summary …", flush=True)

df_features = df_nl.drop(columns=["nl_profile"], errors="ignore")
df_full = df_pidp.merge(df_features, on="pidp", how="left")

summary_rows = []
gb_cols = ["ladcd", "group"] if hier_col else ["ladcd"]
for keys, grp in df_full.groupby(gb_cols):
    ladcd_val, *rest = keys if isinstance(keys, tuple) else (keys,)
    group_val = rest[0] if rest else None

    la_summary = _cs.make_cluster_summary(grp, "cluster", wave=WAVE)

    name_map = (
        df_pidp[
            (df_pidp["ladcd"] == ladcd_val) &
            (df_pidp["group"] == group_val if group_val is not None else True)
        ]
        .set_index("cluster")["tribe_label"]
        .to_dict()
    )
    la_summary["tribe_label"] = la_summary["cluster_id"].map(name_map).fillna(la_summary["tribe_label"])

    la_summary["ladcd"] = ladcd_val
    la_summary["ladnm"] = grp["ladnm"].iloc[0]
    if hier_col:
        la_summary["group"] = group_val
    summary_rows.append(la_summary)

df_summary = pd.concat(summary_rows, ignore_index=True)

leading = ["ladcd", "ladnm"] + (["group"] if hier_col else [])
df_summary = df_summary[leading + [c for c in df_summary.columns if c not in leading]]

df_summary.to_csv(API_OUT, index=False)
print(f"\nSaved {len(df_summary)} cluster rows → {API_OUT}")

preview_cols = leading + ["cluster_id", "tribe_label", "n_respondents", "size"]
print(df_summary[[c for c in preview_cols if c in df_summary.columns]].to_string(index=False))


WAVE=k  HIERARCHICAL_CLUSTER=jbstat_eng  MODEL=gpt-4o
COVERAGE_TARGET=80%  MAX_PIDPS_THRESHOLD=2000  MAX_PROFILES_PER_CALL=100
OpenAI client ready.

Loading NL profiles …
  27,330 respondents loaded.
  Profile columns available: 9/9
Loading LA counts …
  7,969,122 rows, 346 LAs loaded.
  TEST MODE — 4 LA(s): ['Hounslow', 'Islington', 'Newham', 'Tower Hamlets']
  Merged: 95,268 rows
  Building compact profile encodings …
  Unique compact profiles: 22,165

── 80% coverage table ──
        ladnm  group  unique_pidps  pidps_80pct
     Hounslow    1.0         14226         5685
     Hounslow    3.0           960          339
     Hounslow    4.0          6252         2895
     Hounslow    5.0          1045          424
     Hounslow    7.0          1433          555
     Hounslow    8.0           953          328
    Islington    1.0         14345         5670
    Islington    3.0          1010          384
    Islington    4.0          5894         2821
    Islington    5.0          1011  